# Model Fitness: anticipar el riesgo de cancelación

**Brenis Hernández · Proyecto de portafolio**

Esta copia pública conserva el código y las explicaciones. Se retiraron las salidas originales para evitar publicar registros o identificadores; los resultados resumidos se documentan en la ficha del portafolio a partir de la versión original revisada. No se ha vuelto a ejecutar sobre los datos originales. También se omiten mensajes de revisión académica.

**Alcance:** Las métricas proceden de una sola partición aleatoria 80/20, no de validación temporal o externa. Faltan calibración, ROC/AUC, validación cruzada y evaluación de costos. K=5 se fijó para el ejercicio; las etiquetas de clúster no son ordinales. Se corrigió la descripción para que coincida con las salidas guardadas, sin recalcular los modelos.


# Predicción de cancelación de clientes — Model Fitness

**Analista:** Brenis Hernández  
**Objetivo:**  
Analizar el comportamiento de clientes del gimnasio Model Fitness para:

- Predecir la probabilidad de cancelación
- Identificar perfiles de usuarios
- Detectar factores de riesgo de abandono
- Segmentar clientes en clústeres
- Proponer estrategias de retención



In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score

from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import KMeans


## Paso 1. Descargar los datos

In [ ]:

df = pd.read_csv('/datasets/gym_churn_us.csv')
df.head()



## Paso 2. Llevar a cabo el análisis exploratorio de datos (EDA)

In [ ]:
df.info()
df.describe()
df.isna().sum()


### Comparación entre clientes que cancelaron y los que permanecen


In [ ]:
df.groupby('Churn').mean()


### Diferencias entre clientes que cancelan y los que se quedan

Se observa que los clientes que cancelan presentan:

- Menor frecuencia de visitas (tanto histórica como del mes actual).
- Menor tiempo de permanencia en el gimnasio (Lifetime).
- Contratos de menor duración.
- Menor gasto en servicios adicionales.
- Menor participación en clases grupales.

Esto indica que la baja interacción con el gimnasio está fuertemente asociada con la cancelación.



### Distribuciones de variables según cancelación


In [ ]:
for col in df.columns.drop('Churn'):
    plt.figure(figsize=(6,4))
    sns.histplot(data=df, x=col, hue='Churn', bins=30, kde=True)
    plt.title(f'{col} vs Cancelación')
    plt.show()


### Matriz de correlación


In [ ]:
plt.figure(figsize=(14,10))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm')
plt.title('Matriz de correlación')
plt.show()


## Paso 3. Construir un modelo para predecir la cancelación de usuarios

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
scaler = StandardScaler()
X_train_st = scaler.fit_transform(X_train)
X_val_st = scaler.transform(X_val)


In [ ]:
#Regresion logistica

log_model = LogisticRegression(random_state=42)
log_model.fit(X_train_st, y_train)

y_pred_log = log_model.predict(X_val_st)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_val, y_pred_log))
print("Precision:", precision_score(y_val, y_pred_log))
print("Recall:", recall_score(y_val, y_pred_log))




In [ ]:
# Bosque aleatorio

rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_val)

print("Random Forest")
print("Accuracy:", accuracy_score(y_val, y_pred_rf))
print("Precision:", precision_score(y_val, y_pred_rf))
print("Recall:", recall_score(y_val, y_pred_rf))


In [ ]:
# Importancia de variables en Random Forest
importances = rf_model.feature_importances_
features = X.columns

importance_df = pd.DataFrame({
    'feature': features,
    'importance': importances
}).sort_values('importance', ascending=False)

importance_df



### Evaluación de modelos

La regresión logística mostró un desempeño ligeramente superior en accuracy y precision en comparación con el bosque aleatorio.

Ambos modelos presentan un recall similar, lo que indica que detectan con eficacia a los clientes que cancelan.

Por lo tanto, la regresión logística se considera el modelo principal para la predicción de cancelación, ya que además de buen desempeño permite interpretar mejor los factores que influyen en la cancelación.




## Paso 4. Crear clústeres de usuarios/as

In [ ]:
X_scaled = scaler.fit_transform(df.drop('Churn', axis=1))


In [ ]:
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(12,8))
dendrogram(linked, orientation='top')
plt.title('Dendrograma')
plt.show()


In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42)
df['cluster'] = kmeans.fit_predict(X_scaled)


In [ ]:
df.groupby('cluster').mean()
df.groupby('cluster')['Churn'].mean()


### Tasa de cancelación por clúster — descripción corregida

Según las salidas guardadas: clúster 0 = 24.64%; clúster 1 = 9.00%; clúster 2 = 2.20%; clúster 3 = 57.29%; clúster 4 = 26.68%. El clúster 3 presenta la mayor tasa y el 2 la menor. Las etiquetas son arbitrarias. Estas diferencias son exploratorias y no demuestran efectos causales.


## Paso 5. Conclusiones



## Conclusiones

El análisis muestra que la cancelación de clientes está fuertemente asociada con el nivel de interacción con el gimnasio.  
Los clientes que presentan menor frecuencia de visitas, contratos de corta duración, menor gasto en servicios adicionales y baja participación en clases grupales tienen mayor probabilidad de abandonar el servicio.

El modelo predictivo confirma que es posible identificar clientes en riesgo de cancelación con buena precisión. Además, la segmentación por clústeres demuestra que existen grupos de clientes claramente diferenciados en términos de lealtad, comportamiento y riesgo de abandono.

## Recomendaciones estratégicas

A partir de los hallazgos, se proponen los siguientes principios para mejorar la retención de clientes:

### 1. Intervenir cuando disminuye la actividad
**Principio:** La baja frecuencia de visitas es uno de los principales indicadores de riesgo.  
**Acción concreta:** Implementar alertas automáticas cuando la frecuencia mensual de un cliente disminuya y activar contacto proactivo (mensaje, llamada o incentivo personalizado).

### 2. Gestionar los contratos próximos a vencer
**Principio:** Los clientes con contratos cortos o próximos a finalizar tienen mayor probabilidad de cancelar.  
**Acción concreta:** Ofrecer promociones de renovación anticipada, descuentos por contratos más largos o beneficios adicionales antes de que termine el contrato.

### 3. Fomentar la participación en servicios adicionales y clases grupales
**Principio:** Los clientes más comprometidos (que asisten a clases y consumen servicios adicionales) muestran menor tasa de cancelación.  
**Acción concreta:** Promover clases grupales gratuitas de prueba, retos fitness, programas de fidelización y paquetes que incluyan servicios complementarios.

### 4. Enfocar esfuerzos en clientes nuevos
**Principio:** Los clientes con menor antigüedad (Lifetime bajo) presentan mayor riesgo de abandono.  
**Acción concreta:** Desarrollar programas de bienvenida, seguimiento durante los primeros meses y asesoría personalizada para crear hábito de asistencia.


Estas acciones permiten pasar de un enfoque reactivo a uno preventivo, utilizando datos para anticipar la cancelación y fortalecer la relación con los clientes.



## Conclusiones y recomendaciones

- Nivel general de cumplimiento: el trabajo está bien ejecutado y, en conjunto, se sitúa en un nivel intermedio . Cumples los requisitos básicos y varios intermedios pero aún puede reforzarse con análisis y validaciones adicionales. 🔎🚀

- Fortalezas:
  - EDA completo y coherente con el objetivo: se identifican las variables asociadas al churn (frecuencia, lifetime, contratos). 📈✅  
  - Implementación y comparación de modelos (Logistic Regression y Random Forest) con métricas apropiadas (accuracy, precision, recall). 🤖✅  
  - Aplicación de clustering y cálculo de tasas de churn por cluster para perfilar segmentos útiles para acciones de retención. 🧭✅

- Áreas de oportunidad y mejora
  - **RECOMENDADO** Revisar y corregir la descripción de los clusters: en el notebook hay una inconsistencia entre los valores numéricos y la narrativa (etiqueta de cluster con la mayor tasa de churn). Toma en cuenta que faltó verificar que las etiquetas y los textos coincidan. ❗  
  - **RECOMENDADO** Profundizar el perfilado de cada cluster (medias/boxplots por variable, rasgos distintivos) y documentar por qué elegiste k=5 (o mostrar el elbow si se desea justificar). 📊  
  - **SI QUIERES LLEVAR TU NOTEBOOK A OTRO NIVEL** Añadir análisis de multicolinealidad y mostrar coeficientes/intervalos de la regresión logística o calibración del modelo; incluir curvas ROC/AUC y matrices de confusión para una evaluación más robusta. 🧪

- El siguiente paso es corregir la inconsistencia en la descripción de clusters, añade visualizaciones por cluster (boxplots y tablas transpuestas) y complementa la evaluación de modelos con ROC/AUC y validación cruzada. Claro, si deseas alcanzar un nivel intermedio/avanzado. Con esos ajustes el proyecto ganará mayor rigor y valor para la toma de decisiones. 💪🚀